In [ ]:
import os
import h5py
import torch
from torch.utils.data import Dataset
import numpy as np
import torch
from torch import optim
import torch.nn as nn
from torch.utils.data import DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
from numpy import random
import cv2
from numpy import identity

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!unzip -q "/content/drive/MyDrive/landslide4sense.zip" -d /content/

In [ ]:
print(os.listdir('/content'))

In [ ]:
def compute_topographical_features(dem, slope, res=10.0):
    """ Compute northness, eastness, profile curvature"""

    dem_padded = np.pad(dem, pad_width=1, mode="edge")

    dy, dx = np.gradient(dem_padded, res)

    d2y, _ = np.gradient(dy, res)
    _, d2x = np.gradient(dx, res)

    # removing the padding added in the beginning
    dx = dx[1:-1, 1:-1]
    dy = dy[1:-1, 1:-1]
    d2x = d2x[1:-1, 1:-1]
    d2y = d2y[1:-1, 1:-1]

    aspect = np.arctan2(-dy, dx)
    northness = np.cos(aspect)
    eastness = np.sin(aspect)

    curvature = d2x + d2y

    return northness, eastness, curvature


def compute_normalization(img_dir, file_ids): 
    """ Compute the mean and the standard deviation from the training set only. Protected against NaN vlaues. """
    N_CHANNELS = 17
    channel_sum = np.zeros(N_CHANNELS, dtype=np.float64)
    channel_squared_sum = np.zeros(N_CHANNELS, dtype=np.float64)
    pixel_count = 0
    eps = 1e-6
    
    for file_id in file_ids:
        img_path = os.path.join(img_dir, f"image_{file_id}.h5")
        if not os.path.exists(img_path):
            continue  # Skip if the file does not exist

        with h5py.File(img_path, "r") as f:
            raw_image = f["img"][:]
            
        blue  = raw_image[:, :, 1].astype(np.float32)
        green = raw_image[:, :, 2].astype(np.float32)
        red   = raw_image[:, :, 3].astype(np.float32)
        b5    = raw_image[:, :, 4].astype(np.float32)
        b6    = raw_image[:, :, 5].astype(np.float32)
        b7    = raw_image[:, :, 6].astype(np.float32)
        nir   = raw_image[:, :, 7].astype(np.float32)
        swir1 = raw_image[:, :, 10].astype(np.float32)
        swir2 = raw_image[:, :, 11].astype(np.float32)
        slope = raw_image[:, :, 12].astype(np.float32)
        dem   = raw_image[:, :, 13].astype(np.float32)
        
        northness, eastness, curvature = compute_topographical_features(dem, slope)
        
        ndvi = (nir - red) / (nir + red + eps)
        bsi = ((swir1 + red) - (nir + blue)) / ((swir1 + red) + (nir + blue) + eps)
        ndwi = (green - nir) / (green + nir + eps)
        
        image_17ch = np.stack(
            [dem, slope, northness, eastness, curvature, blue, green, red, nir, b5, b6, b7, swir1, swir2, ndvi, bsi, ndwi], axis=-1
        )  # final 17 channel raster
        
        image_17ch = np.nan_to_num(image_17ch, nan=0.0)  # Replace NaN values with 0.0
        
        h, w, _ = image_17ch.shape
        
        channel_sum += np.sum(image_17ch, axis=(0, 1))
        channel_squared_sum += np.sum(image_17ch ** 2, axis=(0, 1))
        pixel_count += h * w

    means = channel_sum / pixel_count
    stds = np.sqrt((channel_squared_sum / pixel_count) - (means ** 2))
    
    return means.astype(np.float32), stds.astype(np.float32)

def train_transform(means, stds):
    return A.Compose([
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),

        A.ShiftScaleRotate(
            shift_limit=0.05,
            scale_limit=(-0.1, 0.1),
            rotate_limit=10,
            border_mode=cv2.BORDER_REFLECT,
            p=0.5
        ),
        # Applying normalization here
        # it does img = (img - mean * max_pixel_value) / (std * max_pixel_value) inder the hood
        A.Normalize(mean=list(means), std=list(stds), max_pixel_value=1.0),

        ToTensorV2()
    ])


def val_transform(means, stds):
    return A.Compose([
        A.Normalize(mean=list(means), std=list(stds), max_pixel_value=1.0),
        ToTensorV2()
    ])

class LandslideDataset(Dataset):

    def __init__(self, img_dir, mask_dir=None, transform=None, file_ids = None):
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.transform = transform

        if file_ids is not None:
            self.file_ids = file_ids
        else:
            self.file_ids = sorted(
                [
                    int(f.split("_")[1].split(".")[0])
                    for f in os.listdir(img_dir)
                    if f.endswith(".h5")
                ]
            )

    def __len__(self):
        return len(self.file_ids)

    def __getitem__(self, idx):
        file_id = self.file_ids[idx]
        img_name = f"image_{file_id}.h5"
        mask_name = f"mask_{file_id}.h5"

        with h5py.File(os.path.join(self.img_dir, img_name), "r") as f:
            raw_image = f["img"][:]

        if self.mask_dir is not None:
            with h5py.File(os.path.join(self.mask_dir, mask_name), "r") as f:
                mask = f["mask"][:]
        else:
            mask = np.zeros((128, 128), dtype=np.int64)

        eps = 1e-6

        blue = raw_image[:, :, 1]
        green = raw_image[:, :, 2]
        red = raw_image[:, :, 3]
        b5 = raw_image[:, :, 4]
        b6 = raw_image[:, :, 5]
        b7 = raw_image[:, :, 6]
        nir = raw_image[:, :, 7]
        # b9 = raw_image[:, :, 8]
        swir1 = raw_image[:, :, 10]
        swir2 = raw_image[:, :, 11]
        
        # Terrain features
        slope = raw_image[:, :, 12]
        dem = raw_image[:, :, 13]

        northness, eastness, curvature = compute_topographical_features(dem, slope)

        ndvi = (nir - red) / (nir + red + eps)

        bsi = ((swir1 + red) - (nir + blue)) / ((swir1 + red) + (nir + blue) + eps)

        ndwi = (green - nir) / (green + nir + eps)

        # axis=-1 means the new axis is added at the END → shape: (128, 128, 17)
        image_17ch = np.stack(
            [dem, slope, northness, eastness, curvature, blue, green, red, nir, b5, b6, b7, swir1, swir2, ndvi, bsi, ndwi], axis=-1
        ).astype(np.float32)  # Final shape: (128, 128, 17)

        image_17ch = np.nan_to_num(image_17ch, nan=0.0, posinf=0.0, neginf=0.0)

    # this will do the normalization, rotation and convert to the tensors
        if self.transform:

            augmented = self.transform(image=image_17ch, mask=mask)
            image = augmented['image'].float()
            mask = augmented['mask'].long()
        else:
            image_17ch = image_17ch.transpose((2, 0, 1))  # (C, H, W)

            image = torch.from_numpy(image_17ch).float()
            mask = torch.from_numpy(mask).long()
            

        return image, mask


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# --------------ResUNet Model----------------------

class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        
        # First convolution
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=False)
        
        # Second convolution
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        # Shortcut connection
        self.shortcut = nn.Sequential()
        if in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        identity = self.shortcut(x)
        
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        
        out = self.conv2(out)
        out = self.bn2(out)
        
        out = out + identity  # Out-of-place addition
        out = self.relu(out)
        
        return out


# ---- Encoder Block ---- #

class EncoderBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = ResidualBlock(in_channels, out_channels)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

    def forward(self, x):
        features = self.conv(x)
        pooled = self.pool(features)
        return features, pooled


# ---- Decoder Block ---- #

class DecoderBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.upsample = nn.ConvTranspose2d(
            in_channels, out_channels, kernel_size=2, stride=2
        )
        self.conv = ResidualBlock(out_channels * 2, out_channels)

    def forward(self, x , skip):
            upsampled = self.upsample(x)
            cat = torch.cat([upsampled, skip], dim=1)
            x = self.conv(cat)
            return x


class ResUNet(nn.Module):
    def __init__(self, in_channels=17, num_classes=2):
        super().__init__()

        # -- Encoding Phase -- 
        self.enc1 = EncoderBlock(in_channels, 64)
        self.enc2 = EncoderBlock(64, 128)
        self.enc3 = EncoderBlock(128, 256)
        self.enc4 = EncoderBlock(256, 512)

        # -- Bottleneck (deepest point - no pooling here) --
        self.bottleneck = ResidualBlock(512, 1024)

        # -- Decoding Phase --
        self.dec4 = DecoderBlock(1024, 512)
        self.dec3 = DecoderBlock(512, 256)
        self.dec2 = DecoderBlock(256, 128)
        self.dec1 = DecoderBlock(128, 64)

        # -- Final Output Layer -- 
        self.output_conv = nn.Conv2d(64, num_classes, kernel_size=1)

    def forward(self, x):
        # --- Encoder ---
        skip1, x = self.enc1(x)
        skip2, x = self.enc2(x)
        skip3, x = self.enc3(x)
        skip4, x = self.enc4(x)

        # --- Bottleneck ---
        x = self.bottleneck(x)

        # --- Decoder ---
        x = self.dec4(x, skip4)
        x = self.dec3(x, skip3)
        x = self.dec2(x, skip2)
        x = self.dec1(x, skip1)

        # Final output
        return self.output_conv(x)

In [ ]:
import torch
import torch.nn as nn


class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth
 
    def forward(self, predictions, targets):
        probs = torch.softmax(predictions, dim=1)[:, 1, :, :] 
        targets_f = targets.float()
 
        intersection = (probs * targets_f).sum(dim=(1, 2))
 
        dice = (2.0 * intersection + self.smooth) / (
            probs.sum(dim=(1, 2)) + targets_f.sum(dim=(1, 2)) + self.smooth
        )
        return 1 - dice.mean()


class BinaryFocalLoss(nn.Module):
    """Focal Loss for binary segmentation (landslide vs background).
       Down-weights easy background pixels and focuses on hard, rare landslide pixels."""
    def __init__(self, alpha, gamma, smooth=1e-6):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.smooth = smooth
        
    def forward(self, logits, targets):
        probs = torch.softmax(logits, dim=1)[:, 1]      # probability of landslide
        targets_f = targets.float()
        
        pt = torch.where(targets_f == 1, probs, 1 - probs)
        
        alpha_t = torch.where(targets_f == 1, self.alpha, 1 - self.alpha)
        
        focal = -alpha_t * (1 - pt) ** self.gamma * torch.log(pt + self.smooth)
        
        return focal.mean()


class CombinedFocalDiceLoss(nn.Module):
    """Blend of Focal Loss (handles class imbalance) and Dice Loss (maximises region overlap).
       Often outperforms CE+Dice for extremely imbalanced tasks like landslide detection."""
    def __init__(self, focal_weight, dice_weight, alpha, gamma):
        super().__init__()
        self.focal = BinaryFocalLoss(alpha=alpha, gamma=gamma)
        self.dice = DiceLoss()
        self.focal_weight = focal_weight
        self.dice_weight = dice_weight

    def forward(self, predictions, targets):
        return (self.focal_weight * self.focal(predictions, targets) +
                self.dice_weight * self.dice(predictions, targets))


class CombinedLoss(nn.Module):
    """Original combined loss: CrossEntropy + Dice.
       CE provides stable gradients early; Dice focuses on landslide overlap.
       This is a solid baseline before switching to Focal+Dice."""
    def __init__(self, dice_weight=0.5, ce_weight=0.5):
        super().__init__()
        self.dice_weight = dice_weight
        self.ce_weight   = ce_weight
        self.dice = DiceLoss()
    
        self.ce   = nn.CrossEntropyLoss()
 
    def forward(self, predictions, targets):

        return (self.ce_weight   * self.ce(predictions, targets) +
                self.dice_weight * self.dice(predictions, targets))

In [ ]:

def compute_metrics(predictions, targets, threshold=0.5):
    """
    predictions : (Batch, 2, H, W) raw logits from the model
    targets     : (Batch, H, W)    ground truth integer labels
    threshold   : float probability threshold for converting landslide probability to binary
    """
    probs = torch.softmax(predictions, dim=1)[:, 1]   # shape: (Batch, H, W)
    pred_bin = probs > threshold


    tp = ((targets == 1) & (pred_bin == 1)).sum().float() 
    fp = ((targets == 0) & (pred_bin == 1)).sum().float()
    fn = ((targets == 1) & (pred_bin == 0)).sum().float()

    return tp, fp, fn

In [ ]:
import os
from ee import image
import torch
import random
import torch.optim as optim
from torch.utils.data import DataLoader

from src.utils import compute_metrics

def train_transfer_learning(
    train_img_dir,
    train_mask_dir, 
    val_img_dir,
    val_mask_dir,
    phase1_epochs,
    phase2_epochs,
    batch_size, 
    pretrained_model_path,
    save_path
):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using {device} for the trainng!!!")
    
    train_files = sorted([f for f in os.listdir(train_img_dir) if f.endswith(".h5")])
    train_ids = [int(f.split("_")[1].split(".")[0]) for f in train_files]
    
    
    val_files = sorted([f for f in os.listdir(val_img_dir) if f.endswith(".h5")])
    all_val_ids = [int(f.split("_")[1].split(".")[0]) for f in val_files]

    random.seed(42)
    random.shuffle(all_val_ids)
    
    split_idx = len(all_val_ids) // 2
    val_ids = all_val_ids[:split_idx]
    test_ids = all_val_ids[split_idx:]
    
    print(f"Train: {len(train_ids)} | Val: {len(val_ids)} | Test: {len(test_ids)}")
    
    print("\n ---- Computing Normalization Statistics From Nepal Training Split ----")
    MEANS, STDS = compute_normalization(train_img_dir, train_ids)
    print(f"Computed Means: {MEANS}")
    print(f"Computed Stds: {STDS}\n")
    
    # Initializing Datasets
    train_dataset = LandslideDataset(train_img_dir, train_mask_dir, transform=train_transform(MEANS, STDS), file_ids=train_ids)
    val_dataset = LandslideDataset(val_img_dir, val_mask_dir, transform=val_transform(MEANS, STDS), file_ids=val_ids)    
    test_dataset =  LandslideDataset(val_img_dir, val_mask_dir, transform=val_transform(MEANS, STDS), file_ids=test_ids)
        
     # Initializing Dataloaders
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)   
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)   
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)   
    
    # -------------- MODEL, LOSS & PRETRAINED WEIGHTS ----------------- #
                        
    model = ResUNet(in_channels=17, num_classes=2).to(device)
    criterion = CombinedFocalDiceLoss(focal_weight=0.35, dice_weight=0.65, alpha=0.50, gamma=2.0)
    
    print(f" Loading pretrained weights from {pretrained_model_path}....")
    checkpoint = torch.load(pretrained_model_path, map_location=device)
    pretrained_dict = checkpoint['model_state_dict']
    
    # ------------ Loading the pretrained model ------------ #
    model_dict = model.state_dict()
    pretrained_dict = {k: v for k, v in pretrained_dict.items() if k in model_dict and v.size() == model_dict[k].size()}
    model_dict.update(pretrained_dict)
    model.load_state_dict(model_dict)
    print(f" [INFO] Loaded {len(pretrained_dict)} matching layer dictionaries.")
    
    best_val_f1 = 0.0 
    
    def run_epoch(epoch, total_epochs, optimizer, scheduler, phase_name):
        nonlocal best_val_f1
        
        model.train()
        running_train_loss = 0.0
        for images, targets in train_loader:
            images, targets = image.to(device), targets.to(device)
            optimizer.zero_grad()
            predictions = model(images)
            loss = criterion(predictions, targets)
            loss.backward()
            optimizer.step()
            running_train_loss = running_train_loss + loss.item()
            
        train_loss = running_train_loss / len(train_loader)
        
        
        # Validation
        model.eval()
        running_val_loss = 0.0
        total_tp, total_fp, total_fn = 0, 0, 0
        
        with torch.no_grad():
            for images, targets in val_loader:
                images, targets = image.to(device), targets.to(device)
                predictions = model(images)
                loss = criterion(predictions, targets)
                running_val_loss = running_val_loss + loss.item()
                
                batch_metrics = compute_metrics(predictions, targets)
                total_tp += batch_metrics[0]
                total_fp += batch_metrics[1]
                total_fn += batch_metrics[2]
                
        val_loss = running_val_loss / len(val_loader)
        iou = total_tp / (total_tp + total_fp + total_fn + 1e-6)
        f1 = 2 * total_tp / (2 * total_tp + total_fp + total_fn + 1e-6)
        
        scheduler.step(1 - iou)
        
        print(
            f"[{phase_name}] Epoch [{epoch:02d}/{total_epochs}] "
            f"| Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val IoU: {iou:.4f} | Val F1: {f1:.4f} "
            f"| LR: {optimizer.param_groups[0]['lr']:.6f}"            
        )
        
        # --- Save Checkpoint ---
        if f1 > best_val_f1:
            best_val_f1 = f1
            torch.save({
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'best_val_f1': best_val_f1
            }, save_path)
            print(f" => Saved new best model checkpoint! F1: {best_val_f1:.4f}")
        
                    
    # ========================================
    # PHASE 1: FREEZE ENCODER & TRAIN DECODER
    # ========================================
    print("\n" + "="*55)
    print("PHASE 1: Feature Extraction (Encoder Frozen) ")
    print("="*55)
    
    # Freeze Encoder leyer
    for name, param in model.named_parameters():
        if 'enc' in name or 'bottleneck' in name:
            param.requires_grad = False
        else:
            param.requires_grad = True # Decoder and output remain active
            
    optimizer_p1 = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4, weight_decay=1e-4)
    scheduler_p1 = optim.lr_scheduler.ReduceLROnPlateau(optimizer_p1, mode="min", patience=3, factor=0.5)
    
    for epoch in range(1, phase1_epochs + 1):
        run_epoch(epoch, phase1_epochs, optimizer_p1, scheduler_p1, "Phase 1")
        
    # ========================================
    # PHASE 2: UNFREEZE ALL AND FINE TUNING
    # ========================================
    
    print("\n" + "="*55)
    print(" PHASE 2: Full Fine-Tuning (All Layers Unfrozen) ")
    print("="*55)
    
    # Unfreeze all the layer
    
    for param in model.parameters():
        param.requires_grad = True
        
    # Much Lower learining rate
    optimizer_p2 = optim.Adam(model.parameters(), lr=1e-5, weight_decay=1e-4)
    scheduler_p2 = optim.lr_scheduler.ReduceLROnPlateau(optimizer_p2, mode="min", patience=5, factor=0.5)
    
    for epoch in range(1, phase2_epochs + 1):
        run_epoch(epoch, phase2_epochs, optimizer_p2, scheduler_p2, "Phase 2")
        
    # ==========================================
    # FINAL PHASE: UNBIASED TEST EVALUATION
    # ==========================================
    print("\n" + "="*55)
    print(" EVALUATING BEST MODEL ON UNSEEN TEST SET ")
    print("="*55)
    
    # loading the best weights found during training
    best_checkpoint = torch.load(save_path)
    model.load_state_dict(best_checkpoint['model_state_dict'])
    model.eval()
    
    test_tp, test_fp, test_fn = 0, 0, 0
    with torch.no_grad():
        for images, targets in test_loader:
            images, targets = images.to(device), targets.to(device)
            predictions = model(images)
            batch_metrics = compute_metrics(predictions, targets)
            test_tp += batch_metrics[0]
            test_fp += batch_metrics[1]
            test_fn += batch_metrics[2]

    test_iou = test_tp / (test_tp + test_fp + test_fn + 1e-6)
    test_f1 = 2 * test_tp / (2 * test_tp + test_fp + test_fn + 1e-6)
    test_precision = test_tp / (test_tp + test_fp + 1e-6)
    test_recall = test_tp / (test_tp + test_fn + 1e-6)

    print(f"Final Test Metrics -> IoU: {test_iou:.4f} | F1: {test_f1:.4f} | Precision: {test_precision:.4f} | Recall: {test_recall:.4f}\n")
    print(f"[SUCCESS] Transfer Learning Pipeline Complete. Model saved to: {save_path}")
    
    return model

In [ ]:
TRAIN_IMG_DIR  = "/content/nepal_dataset/TrainData/img"
TRAIN_MASK_DIR = "/content/nepal_dataset/TrainData/mask"
VAL_IMG_DIR    = "/content/nepal_dataset/ValData/img"
VAL_MASK_DIR   = "/content/nepal_dataset/ValData/mask"

PRETRAINED_MODEL = "/content/landslide4sense_model.pth"
NEW_SAVE_PATH    = "/content/nepal_transfer_learning_model.pth" 

trained_model = train_transfer_learning(
    train_img_dir         = TRAIN_IMG_DIR,
    train_mask_dir        = TRAIN_MASK_DIR,
    val_img_dir           = VAL_IMG_DIR,
    val_mask_dir          = VAL_MASK_DIR,
    phase1_epochs         = 15,
    phase2_epochs         = 60, 
    batch_size            = 16,
    pretrained_model_path = PRETRAINED_MODEL,
    save_path             = NEW_SAVE_PATH
)